In [1]:
import polars as pl

# 1. Caminhos dos arquivos (Configurados para desligamento_faixaetaria)
arquivo_csv = r"C:\Users\MICRO\OneDrive\Documentos\UC2\desligamento_faixaetaria.csv"
arquivo_parquet = r"C:\Users\MICRO\OneDrive\Documentos\UC2\desligamento_faixaetaria.parquet"

# 2. Lendo o CSV informando que o separador é ponto e vírgula
df = pl.read_csv(arquivo_csv, separator=";")

# 3. Salvando o arquivo no formato Parquet
df.write_parquet(arquivo_parquet)

# 4. Configuração para exibir todas as linhas no print e validar a gravação
with pl.Config(tbl_rows=10000):
    stats_query = pl.scan_parquet(arquivo_parquet)
    print(stats_query.collect())


shape: (8_321, 9)
┌────────────┬───────┬────────────┬────────────┬───┬────────────┬────────┬────────────┬────────────┐
│ Data_Desli ┆ Idade ┆ Faixa_Etar ┆ Gênero     ┆ … ┆ Região     ┆ Estado ┆ Motivo_Des ┆ Tempo_Casa │
│ gamento    ┆ ---   ┆ ia         ┆ ---        ┆   ┆ ---        ┆ ---    ┆ ligamento  ┆ _Anos      │
│ ---        ┆ i64   ┆ ---        ┆ str        ┆   ┆ str        ┆ str    ┆ ---        ┆ ---        │
│ str        ┆       ┆ str        ┆            ┆   ┆            ┆        ┆ str        ┆ f64        │
╞════════════╪═══════╪════════════╪════════════╪═══╪════════════╪════════╪════════════╪════════════╡
│ 06/08/2025 ┆ 49    ┆ 45-54 anos ┆ Masculino  ┆ … ┆ Nordeste   ┆ PI     ┆ Sem Justa  ┆ 8.3        │
│            ┆       ┆            ┆            ┆   ┆            ┆        ┆ Causa      ┆            │
│ 15/11/2025 ┆ 29    ┆ 25-34 anos ┆ Feminino   ┆ … ┆ Centro-Oes ┆ MT     ┆ Sem Justa  ┆ 2.0        │
│            ┆       ┆            ┆            ┆   ┆ te         ┆        

In [2]:
import polars as pl

# 1. Caminho do arquivo Parquet de faixa etária
arquivo_parquet = r"C:\Users\MICRO\OneDrive\Documentos\UC2\desligamento_faixaetaria.parquet"

# 2. Criando a consulta combinando Idade, Gênero e Estado
# NOTA: Ajuste o nome das colunas se o seu arquivo usar termos diferentes
query_desligamentos = (
    pl.scan_parquet(arquivo_parquet)
    .group_by(["Idade", "Gênero", "Estado"])
    .agg(pl.len().alias("total_desligamentos"))
    .sort("total_desligamentos", descending=True)
)

# 3. Executando a consulta e exibindo o resultado
with pl.Config(tbl_rows=100): # Exibe até 100 combinações no terminal
    print(query_desligamentos.collect())


shape: (2_282, 4)
┌───────┬─────────────┬────────┬─────────────────────┐
│ Idade ┆ Gênero      ┆ Estado ┆ total_desligamentos │
│ ---   ┆ ---         ┆ ---    ┆ ---                 │
│ i64   ┆ str         ┆ str    ┆ u32                 │
╞═══════╪═════════════╪════════╪═════════════════════╡
│ 35    ┆ Feminino    ┆ ES     ┆ 23                  │
│ 26    ┆ Feminino    ┆ SP     ┆ 19                  │
│ 28    ┆ Feminino    ┆ RJ     ┆ 18                  │
│ 34    ┆ Masculino   ┆ ES     ┆ 18                  │
│ 53    ┆ Masculino   ┆ RJ     ┆ 18                  │
│ 52    ┆ Masculino   ┆ RJ     ┆ 17                  │
│ 25    ┆ Feminino    ┆ ES     ┆ 17                  │
│ 55    ┆ Masculino   ┆ MG     ┆ 17                  │
│ 33    ┆ Masculino   ┆ MG     ┆ 16                  │
│ 51    ┆ Feminino    ┆ SP     ┆ 16                  │
│ 43    ┆ Feminino    ┆ SP     ┆ 16                  │
│ 42    ┆ Masculino   ┆ MG     ┆ 16                  │
│ 62    ┆ Feminino    ┆ ES     ┆ 16            

In [6]:
import polars as pl

# 1. Carregar o arquivo Parquet em modo Lazy
lf = pl.scan_parquet("C:/Users/MICRO/OneDrive/Documentos/UC2/desligamento_faixaetaria.parquet")

# 2. Criar a consulta agrupando por Estado e Gênero
query_tempo_casa = (
    lf.group_by(["Estado", "Gênero"])
    .agg(
        # Encontra o maior tempo de casa (em anos) para cada grupo
        pl.col("Tempo_Casa_Anos").max().alias("Maior_Tempo_Casa_Anos"),
        # Opcional: calcula também a média de tempo de casa do grupo
        pl.col("Tempo_Casa_Anos").mean().round(1).alias("Media_Tempo_Casa_Anos")
    )
    # Ordena do maior tempo de casa para o menor
    .sort("Maior_Tempo_Casa_Anos", descending=True)
)

# 3. Executar e salvar o resultado
caminho_saida = "C:/Users/MICRO/OneDrive/Documentos/UC2/maior_tempo_casa_estado_genero.csv"
query_tempo_casa.sink_csv(caminho_saida)

# 4. Exibir o topo do resultado direto no terminal para você ver a resposta
print(query_tempo_casa.limit(1).collect())






shape: (1, 4)
┌────────┬──────────┬───────────────────────┬───────────────────────┐
│ Estado ┆ Gênero   ┆ Maior_Tempo_Casa_Anos ┆ Media_Tempo_Casa_Anos │
│ ---    ┆ ---      ┆ ---                   ┆ ---                   │
│ str    ┆ str      ┆ f64                   ┆ f64                   │
╞════════╪══════════╪═══════════════════════╪═══════════════════════╡
│ ES     ┆ Feminino ┆ 25.0                  ┆ 4.2                   │
└────────┴──────────┴───────────────────────┴───────────────────────┘


In [7]:
import polars as pl

# 1. Carregar o arquivo Parquet em modo Lazy
lf = pl.scan_parquet("C:/Users/MICRO/OneDrive/Documentos/UC2/desligamento_faixaetaria.parquet")

# 2. Criar a consulta para contar os desligamentos por Gênero, Estado e Idade
query_ranking_desligamentos = (
    lf.group_by(["Gênero", "Estado", "Idade"])
    .agg(
        # Conta o total de registros (desligamentos) para cada combinação
        pl.len().alias("Total_Desligamentos")
    )
    # Ordena do maior número de desligamentos para o menor
    .sort("Total_Desligamentos", descending=True)
)

# 3. Executar e salvar o ranking completo em um arquivo CSV
caminho_saida = "C:/Users/MICRO/OneDrive/Documentos/UC2/ranking_desligamentos.csv"
query_ranking_desligamentos.sink_csv(caminho_saida)

# 4. Exibir o topo da lista (o grupo que mais teve desligamentos) na sua tela
print("\n--- GRUPO COM MAIS DESLIGAMENTOS ---")
print(query_ranking_desligamentos.limit(5).collect())



--- GRUPO COM MAIS DESLIGAMENTOS ---
shape: (5, 4)
┌───────────┬────────┬───────┬─────────────────────┐
│ Gênero    ┆ Estado ┆ Idade ┆ Total_Desligamentos │
│ ---       ┆ ---    ┆ ---   ┆ ---                 │
│ str       ┆ str    ┆ i64   ┆ u32                 │
╞═══════════╪════════╪═══════╪═════════════════════╡
│ Feminino  ┆ ES     ┆ 35    ┆ 23                  │
│ Feminino  ┆ SP     ┆ 26    ┆ 19                  │
│ Masculino ┆ ES     ┆ 34    ┆ 18                  │
│ Feminino  ┆ RJ     ┆ 28    ┆ 18                  │
│ Masculino ┆ RJ     ┆ 53    ┆ 18                  │
└───────────┴────────┴───────┴─────────────────────┘


In [8]:
import polars as pl

# 1. Carregar o arquivo Parquet em modo Lazy
lf = pl.scan_parquet("C:/Users/MICRO/OneDrive/Documentos/UC2/desligamento_faixaetaria.parquet")

# 2. Criar a consulta agrupando pela Faixa Etária
query_faixa_campeã = (
    lf.group_by("Faixa_Etaria")
    .agg(
        # Conta o total de demissões/saídas por faixa
        pl.len().alias("Total_Desligamentos")
    )
    # Ordena da faixa com mais desligamentos para a com menos
    .sort("Total_Desligamentos", descending=True)
)

# 3. Executar e salvar o resultado
caminho_saida = "C:/Users/MICRO/OneDrive/Documentos/UC2/desligamentos_por_faixa_etaria.csv"
query_faixa_campeã.sink_csv(caminho_saida)

# 4. Exibir o ranking completo na tela
print("\n--- RANKING DE DESLIGAMENTOS POR FAIXA ETÁRIA ---")
print(query_faixa_campeã.collect())



--- RANKING DE DESLIGAMENTOS POR FAIXA ETÁRIA ---
shape: (5, 2)
┌─────────────────┬─────────────────────┐
│ Faixa_Etaria    ┆ Total_Desligamentos │
│ ---             ┆ ---                 │
│ str             ┆ u32                 │
╞═════════════════╪═════════════════════╡
│ 55 anos ou mais ┆ 1915                │
│ 35-44 anos      ┆ 1756                │
│ 45-54 anos      ┆ 1737                │
│ 25-34 anos      ┆ 1696                │
│ 18-24 anos      ┆ 1217                │
└─────────────────┴─────────────────────┘


In [9]:
import polars as pl

# 1. Carregar o arquivo Parquet em modo Lazy
lf = pl.scan_parquet("C:/Users/MICRO/OneDrive/Documentos/UC2/desligamento_faixaetaria.parquet")

# 2. Criar a consulta filtrando por Justa Causa e agrupando por Região
query_justa_causa = (
    lf.filter(
        # Filtra apenas as linhas onde o motivo seja exatamente Justa Causa
        # (Ajuste o texto "Justa Causa" caso na sua base esteja escrito diferente, ex: "Demissão por Justa Causa")
        pl.col("Motivo_Desligamento").str.contains("Justa Causa")
    )
    .group_by("Região")
    .agg(
        # Conta a quantidade de demissões por justa causa em cada região
        pl.len().alias("Total_Justa_Causa")
    )
    # Ordena da região com mais casos para a com menos
    .sort("Total_Justa_Causa", descending=True)
)

# 3. Executar e salvar o resultado em um arquivo CSV
caminho_saida = "C:/Users/MICRO/OneDrive/Documentos/UC2/justa_causa_por_regiao.csv"
query_justa_causa.sink_csv(caminho_saida)

# 4. Exibir o ranking na tela
print("\n--- RANKING DE JUSTA CAUSA POR REGIÃO ---")
print(query_justa_causa.collect())



--- RANKING DE JUSTA CAUSA POR REGIÃO ---
shape: (5, 2)
┌──────────────┬───────────────────┐
│ Região       ┆ Total_Justa_Causa │
│ ---          ┆ ---               │
│ str          ┆ u32               │
╞══════════════╪═══════════════════╡
│ Sudeste      ┆ 1871              │
│ Nordeste     ┆ 978               │
│ Sul          ┆ 601               │
│ Centro-Oeste ┆ 418               │
│ Norte        ┆ 212               │
└──────────────┴───────────────────┘
